In [2]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Anchor to project root and connect to the database
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebook' else Path.cwd()
DB_PATH = PROJECT_ROOT / 'data' / 'clinical_trials.db'

conn = sqlite3.connect(DB_PATH)

# Load each table into a pandas DataFrame for profiling
studies = pd.read_sql("SELECT * FROM studies", conn)
conditions = pd.read_sql("SELECT * FROM conditions", conn)
interventions = pd.read_sql("SELECT * FROM interventions", conn)
outcomes = pd.read_sql("SELECT * FROM outcomes", conn)
sponsors = pd.read_sql("SELECT * FROM sponsors", conn)
locations = pd.read_sql("SELECT * FROM locations", conn)
study_design = pd.read_sql("SELECT * FROM study_design", conn)

print("Tables loaded:")
print(f"  studies:       {len(studies)} rows")
print(f"  conditions:    {len(conditions)} rows")
print(f"  interventions: {len(interventions)} rows")
print(f"  outcomes:      {len(outcomes)} rows")
print(f"  sponsors:      {len(sponsors)} rows")
print(f"  locations:     {len(locations)} rows")
print(f"  study_design:  {len(study_design)} rows")

Tables loaded:
  studies:       500 rows
  conditions:    864 rows
  interventions: 807 rows
  outcomes:      3244 rows
  sponsors:      824 rows
  locations:     2791 rows
  study_design:  500 rows


In [8]:
studies.head()

,study_id,nct_id,title,acronym,status,phase,study_type,start_date,completion_date,primary_completion_date,enrollment,enrollment_type,brief_summary,eligibility_criteria,minimum_age,maximum_age,gender,created_at,updated_at
0,1,NCT04483635,PRevention of COVID-19 With Oral Vitamin D Sup...,PROTECT,TERMINATED,PHASE3,INTERVENTIONAL,2021-02-08,2021-05-25,2021-05-04,34.0,ACTUAL,"In this 16-week randomized control study, heal...",Inclusion Criteria:\n\n* are aged ≥18 and \<70...,None,None,ALL,2026-07-14 10:28:31,2026-07-14 10:28:31
1,2,NCT04269135,"Fit, Healthy and Smart Preschool Children: PRE...",PREFIT-Chile,COMPLETED,NaN,OBSERVATIONAL,2019-06-01,2020-03-30,2020-03-30,868.0,ACTUAL,Introduction: Investigating the interrelations...,"Inclusion Criteria:\n\n* Preschool child, 4 to...",None,None,ALL,2026-07-14 10:28:31,2026-07-14 10:28:31
2,3,NCT01705535,The Effect of Physiotherapy for the Treatment ...,NaN,UNKNOWN,NA,INTERVENTIONAL,2012-10,2019-07,2019-07,102.0,ACTUAL,Fecal incontinence is the complaint of involun...,Inclusion Criteria:\n\n* Patients refered to e...,None,None,ALL,2026-07-14 10:28:31,2026-07-14 10:28:31
3,4,NCT00358735,Deep Vein Thrombosis (DVT) Prevention in Total...,SAFE,COMPLETED,NA,INTERVENTIONAL,2006-06,2008-12,2008-09,411.0,ACTUAL,Evaluation of the safety and effectiveness of ...,Inclusion Criteria:\n\nAdult patient (Age \>18...,None,None,ALL,2026-07-14 10:28:31,2026-07-14 10:28:31
4,5,NCT06057935,A Study of Additional Chemotherapy After Surge...,NaN,RECRUITING,PHASE2,INTERVENTIONAL,2023-09-21,2028-09-21,2028-09-21,64.0,ESTIMATED,The purpose of this study is to find out wheth...,Inclusion Criteria:\n\n* Patient age 18 years ...,None,None,ALL,2026-07-14 10:28:31,2026-07-14 10:28:31


In [9]:
studies.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   study_id                 500 non-null    int64  
 1   nct_id                   500 non-null    str    
 2   title                    500 non-null    str    
 3   acronym                  127 non-null    str    
 4   status                   500 non-null    str    
 5   phase                    399 non-null    str    
 6   study_type               499 non-null    str    
 7   start_date               497 non-null    str    
 8   completion_date          484 non-null    str    
 9   primary_completion_date  487 non-null    str    
 10  enrollment               493 non-null    float64
 11  enrollment_type          490 non-null    str    
 12  brief_summary            499 non-null    str    
 13  eligibility_criteria     499 non-null    str    
 14  minimum_age              0 non-null  

In [10]:
# Completeness: null count and percentage per column, sorted worst-first
null_counts = studies.isnull().sum()
null_pct = (studies.isnull().mean() * 100).round(1)

completeness = pd.DataFrame({
    'missing_count': null_counts,
    'missing_pct': null_pct
}).sort_values('missing_pct', ascending=False)

completeness[completeness['missing_count'] > 0]   # show only columns with gaps

,missing_count,missing_pct
maximum_age,500,100.0
minimum_age,500,100.0
acronym,373,74.6
phase,101,20.2
completion_date,16,3.2
primary_completion_date,13,2.6
enrollment_type,10,2.0
enrollment,7,1.4
start_date,3,0.6
gender,3,0.6


## Profiling Findings — Completeness (Dimension 1)

| Column | Missing | % | Severity | Note |
|--------|---------|---|----------|------|
| minimum_age | 500 | 100% | Medium | API provides stdAges categories, not numeric ages — column unfillable |
| maximum_age | 500 | 100% | Medium | Same as above |
| acronym | 373 | 74.6% | Low | Optional metadata, not analytically important |
| phase | 101 | 20.2% | Medium-High | Impacts phase analysis; partly expected (observational studies have no phase) — investigate |
| completion_date | 16 | 3.2% | Medium | Impacts duration/completion analysis |
| enrollment | 7 | 1.4% | Medium | Impacts enrollment analysis (Q4) |
| enrollment_type | 10 | 2.0% | Low | |
| gender, start_date | 3 each | 0.6% | Low | |

In [11]:
%%sql
Select studies.status, count(*) from studies group by status

,status,count(*)
0,ACTIVE_NOT_RECRUITING,16
1,AVAILABLE,1
2,COMPLETED,268
3,ENROLLING_BY_INVITATION,5
4,NOT_YET_RECRUITING,16
5,RECRUITING,62
6,SUSPENDED,1
7,TERMINATED,27
8,UNKNOWN,84
9,WITHDRAWN,19


In [12]:
# Cardinality: how many distinct values does each column have?
cardinality = studies.nunique().sort_values()
print("Distinct values per column:\n")
print(cardinality)

Distinct values per column:

maximum_age                  0
minimum_age                  0
updated_at                   1
created_at                   1
enrollment_type              2
gender                       3
study_type                   3
phase                        6
status                      11
acronym                    126
enrollment                 209
primary_completion_date    381
completion_date            385
start_date                 416
eligibility_criteria       498
brief_summary              499
study_id                   500
title                      500
nct_id                     500
dtype: int64


In [29]:
# Value distributions for key categorical columns
for col in ['status', 'phase', 'study_type', 'gender', 'enrollment_type']:
    counts = studies[col].value_counts(dropna=False)
    percentages = (counts/counts.sum()) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'percentage': percentages.map('{:.2f}%'.format)
    })
    print(f"\n===== {col} =====")
    print(summary_df)


===== status =====
                         Count percentage
status                                   
COMPLETED                  268     53.60%
UNKNOWN                     84     16.80%
RECRUITING                  62     12.40%
TERMINATED                  27      5.40%
WITHDRAWN                   19      3.80%
NOT_YET_RECRUITING          16      3.20%
ACTIVE_NOT_RECRUITING       16      3.20%
ENROLLING_BY_INVITATION      5      1.00%
WITHHELD                     1      0.20%
SUSPENDED                    1      0.20%
AVAILABLE                    1      0.20%

===== phase =====
              Count percentage
phase                         
NA              198     39.60%
NaN             101     20.20%
PHASE1           66     13.20%
PHASE2           52     10.40%
PHASE4           43      8.60%
PHASE3           37      7.40%
EARLY_PHASE1      3      0.60%

===== study_type =====
                 Count percentage
study_type                       
INTERVENTIONAL     399     79.80%
OBSERVATIO

In [46]:
%%sql

SELECT
    study_type,
    COUNT(*)                                          AS total,
    SUM(CASE WHEN phase IS NOT NULL THEN 1 ELSE 0 END) AS has_phase_field,
    SUM(CASE WHEN phase IS NULL THEN 1 ELSE 0 END)     AS missing_phase,
    SUM(CASE WHEN phase = 'NA' THEN 1 ELSE 0 END)      AS phase_is_NA_value,
    SUM(CASE WHEN phase IS NOT NULL AND phase != 'NA' THEN 1 ELSE 0 END) AS real_phase
FROM studies
GROUP BY study_type
ORDER BY total DESC


,study_type,total,has_phase_field,missing_phase,phase_is_NA_value,real_phase
0,INTERVENTIONAL,399,399,0,198,201
1,OBSERVATIONAL,99,0,99,0,0
2,EXPANDED_ACCESS,1,0,1,0,0
3,NaN,1,0,1,0,0


In [31]:
# Is missing phase explained by study type?
print("Phase availability by study type:\n")
phase_by_type = studies.groupby('study_type')['phase'].apply(
    lambda x: pd.Series({
        'total': len(x),
        'has_phase': x.notna().sum(),
        'missing_phase': x.isna().sum(),
        'phase_NA_value': (x == 'NA').sum()
    })
).unstack()
print(phase_by_type)

Phase availability by study type:

                 total  has_phase  missing_phase  phase_NA_value
study_type                                                      
EXPANDED_ACCESS      1          0              1               0
INTERVENTIONAL     399        399              0             198
OBSERVATIONAL       99          0             99               0


### Phase completeness — investigated, reclassified

**Initial observation:** phase missing for 101 studies (20.2%).

**Investigation (SQL, grouped by study_type):** all 101 missing phases belong to
non-interventional studies (99 observational, 1 expanded access, 1 unknown type).
Zero interventional studies are missing phase.

**Conclusion:** NOT a completeness defect. Phase is an interventional-trial concept;
observational studies have no phase by design. Reclassified as **expected structural null**.

**Related finding:** Among 399 interventional studies, 198 (49.6%) have phase = "NA"
(explicitly not applicable — e.g. device/behavioral trials outside the Phase 1-4 framework).

**Net analytical constraint:** only 201 of 500 studies (40.2%) have an analyzable phase.
Phase-based analysis (Q2, Q5) must be scoped to these, and this limitation stated explicitly.

In [3]:
query = """
SELECT
    LENGTH(start_date)          AS date_length,
    COUNT(*)                    AS count,
    MIN(start_date)             AS example_min,
    MAX(start_date)             AS example_max
FROM studies
WHERE start_date IS NOT NULL
GROUP BY LENGTH(start_date)
ORDER BY date_length
"""
pd.read_sql(query, conn)

,date_length,count,example_min,example_max
0,7,173,1973-06,2026-09
1,10,324,1983-02-03,2027-01-01


In [17]:
query = """
SELECT 'start_date' AS column_name,
       LENGTH(start_date) AS date_length,
       COUNT(*) AS count
FROM studies WHERE start_date IS NOT NULL
GROUP BY LENGTH(start_date)

    UNION ALL

SELECT 'completion_date',
       LENGTH(completion_date),
       COUNT(*)
FROM studies WHERE completion_date IS NOT NULL
GROUP BY LENGTH(completion_date)

UNION ALL

SELECT 'primary_completion_date',
       LENGTH(primary_completion_date),
       COUNT(*)
FROM studies WHERE primary_completion_date IS NOT NULL
GROUP BY LENGTH(primary_completion_date)

ORDER BY column_name, date_length
"""
pd.read_sql(query, conn)

,column_name,date_length,count
0,completion_date,7,181
1,completion_date,10,303
2,primary_completion_date,7,189
3,primary_completion_date,10,298
4,start_date,7,173
5,start_date,10,324


In [7]:
query = """
SELECT
    COUNT(*)                                              AS total_with_enrollment,
    SUM(CASE WHEN enrollment < 0 THEN 1 ELSE 0 END)       AS negative_values,
    SUM(CASE WHEN enrollment = 0 THEN 1 ELSE 0 END)       AS zero_values,
    MIN(enrollment)                                        AS min_enrollment,
    MAX(enrollment)                                        AS max_enrollment,
    ROUND(AVG(enrollment), 1)                              AS avg_enrollment
FROM studies
WHERE enrollment IS NOT NULL
"""
pd.read_sql(query, conn)

,total_with_enrollment,negative_values,zero_values,min_enrollment,max_enrollment,avg_enrollment
0,493,0,19,0,800000,2121.5


In [16]:
query = """
Select
    status,
    count(*)
from studies
    where enrollment  = 0
group by status
"""
pd.read_sql(query, conn)

,status,count(*)
0,WITHDRAWN,19


In [9]:
%%sql
SELECT status, COUNT(*) AS count, MIN(start_date) AS earliest, MAX(start_date) AS latest
FROM studies
WHERE start_date > '2026-07'
GROUP BY status
ORDER BY count DESC

,status,count,earliest,latest
0,NOT_YET_RECRUITING,4,2026-07-01,2027-01-01


In [2]:
query = """
SELECT nct_id, status, study_type, phase, enrollment, title
FROM studies
WHERE enrollment IS NOT NULL
ORDER BY enrollment DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,nct_id,status,study_type,phase,enrollment,title
0,NCT02014610,UNKNOWN,OBSERVATIONAL,NaN,800000,White Blood Cell Counts and Onset of Cardiovas...
1,NCT03900910,RECRUITING,INTERVENTIONAL,NA,50000,Gastric Cancer Prevention for Indigenous Peoples
2,NCT00125008,COMPLETED,INTERVENTIONAL,PHASE4,37673,Evaluation of the Vi Polysaccharide Vaccine Ag...
3,NCT03359876,COMPLETED,OBSERVATIONAL,NaN,16000,Clinical Outcomes Among Non-valvular Atrial Fi...
4,NCT06117618,COMPLETED,INTERVENTIONAL,NA,12284,Sepsis Electronic Prompting for Timely Interve...
5,NCT07448090,RECRUITING,OBSERVATIONAL,NaN,10000,Real-World Case Registry Study on Obesity
6,NCT04212520,RECRUITING,OBSERVATIONAL,NaN,6500,Predictive Value of Non-fasting Blood Lipid Le...
7,NCT00341835,COMPLETED,OBSERVATIONAL,NaN,6356,Genetic Epidemiology of Lung Cancer
8,NCT01891240,UNKNOWN,OBSERVATIONAL,NaN,5000,IMproved PRegnancy Outcome by Early Detection
9,NCT05009225,UNKNOWN,OBSERVATIONAL,NaN,4000,Clinical Decision Support for Atrial Fibrillat...


In [2]:
query = """
SELECT
    COUNT(*)                    AS total_rows,
    COUNT(DISTINCT nct_id)      AS distinct_nct_ids,
    COUNT(*) - COUNT(DISTINCT nct_id) AS duplicate_count
FROM studies
"""
pd.read_sql(query, conn)

,total_rows,distinct_nct_ids,duplicate_count
0,500,500,0


In [3]:
query = """
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN LENGTH(nct_id) != 11 THEN 1 ELSE 0 END)           AS wrong_length,
    SUM(CASE WHEN nct_id NOT LIKE 'NCT%' THEN 1 ELSE 0 END)         AS missing_prefix,
    SUM(CASE WHEN LENGTH(nct_id) = 11
              AND nct_id LIKE 'NCT%'
             THEN 0 ELSE 1 END)                                      AS potentially_malformed
FROM studies
"""
pd.read_sql(query, conn)

,total,wrong_length,missing_prefix,potentially_malformed
0,500,0,0,0


In [4]:
import re

# Proper format validation: NCT followed by exactly 8 digits
pattern = re.compile(r'^NCT\d{8}$')

studies['nct_id_valid'] = studies['nct_id'].apply(
    lambda x: bool(pattern.match(str(x))) if pd.notna(x) else False
)

invalid_ids = studies[~studies['nct_id_valid']]

print(f"Total studies:        {len(studies)}")
print(f"Valid NCT format:     {studies['nct_id_valid'].sum()}")
print(f"Invalid NCT format:   {len(invalid_ids)}")

if len(invalid_ids) > 0:
    print("\nMalformed NCT IDs found:")
    print(invalid_ids[['study_id', 'nct_id', 'status']])

Total studies:        500
Valid NCT format:     500
Invalid NCT format:   0


In [5]:
%%sql
SELECT s.nct_id, c.condition_name
FROM studies s
JOIN conditions c ON s.study_id = c.study_id

,nct_id,condition_name
0,NCT04483635,COVID-19
1,NCT04269135,"Child, Preschool"
2,NCT01705535,Fecal Incontinence
3,NCT00358735,Deep Vein Thrombosis of Lower Limb
4,NCT00358735,Pulmonary Embolism (PE)
...,...,...
859,NCT04285359,Infection Post Op
860,NCT02867059,Plasmodium Falciparum Malaria
861,NCT07537959,Hepatocellular Carcinoma (HCC)
862,NCT07537959,Metastasis


In [12]:
query = """
SELECT 'conditions' AS child_table, COUNT(*) AS orphaned_rows
FROM conditions c
LEFT JOIN studies s ON c.study_id = s.study_id
WHERE s.study_id IS NULL

UNION ALL

SELECT 'interventions', COUNT(*)
FROM interventions i
LEFT JOIN studies s ON i.study_id = s.study_id
WHERE s.study_id IS NULL

UNION ALL

SELECT 'outcomes', COUNT(*)
FROM outcomes o
LEFT JOIN studies s ON o.study_id = s.study_id
WHERE s.study_id IS NULL

UNION ALL

SELECT 'sponsors', COUNT(*)
FROM sponsors sp
LEFT JOIN studies s ON sp.study_id = s.study_id
WHERE s.study_id IS NULL

UNION ALL

SELECT 'locations', COUNT(*)
FROM locations l
LEFT JOIN studies s ON l.study_id = s.study_id
WHERE s.study_id IS NULL

UNION ALL

SELECT 'study_design', COUNT(*)
FROM study_design d
LEFT JOIN studies s ON d.study_id = s.study_id
WHERE s.study_id IS NULL
"""
pd.read_sql(query, conn)

,child_table,orphaned_rows
0,conditions,0
1,interventions,0
2,outcomes,0
3,sponsors,0
4,locations,0
5,study_design,0


In [13]:
query = """
SELECT
    (SELECT COUNT(*) FROM studies) AS total_studies,
    (SELECT COUNT(DISTINCT study_id) FROM conditions) AS studies_with_conditions,
    (SELECT COUNT(DISTINCT study_id) FROM locations) AS studies_with_locations,
    (SELECT COUNT(DISTINCT study_id) FROM sponsors) AS studies_with_sponsors,
    (SELECT COUNT(DISTINCT study_id) FROM outcomes) AS studies_with_outcomes,
    (SELECT COUNT(DISTINCT study_id) FROM interventions) AS studies_with_interventions,
    (SELECT COUNT(DISTINCT study_id) FROM study_design) AS studies_with_design
"""
pd.read_sql(query, conn)

,total_studies,studies_with_conditions,studies_with_locations,studies_with_sponsors,studies_with_outcomes,studies_with_interventions,studies_with_design
0,500,499,455,500,486,456,500


In [14]:
query = """
SELECT
    s.study_type,
    COUNT(*) AS total_studies,
    SUM(CASE WHEN i.study_id IS NULL THEN 1 ELSE 0 END) AS studies_without_interventions
FROM studies s
LEFT JOIN (SELECT DISTINCT study_id FROM interventions) i
    ON s.study_id = i.study_id
GROUP BY s.study_type
"""
pd.read_sql(query, conn)

,study_type,total_studies,studies_without_interventions
0,NaN,1,1
1,EXPANDED_ACCESS,1,0
2,INTERVENTIONAL,399,0
3,OBSERVATIONAL,99,43


In [15]:
query = """
SELECT
    s.status,
    COUNT(*) AS total,
    SUM(CASE WHEN l.study_id IS NULL THEN 1 ELSE 0 END) AS without_locations,
    ROUND(100.0 * SUM(CASE WHEN l.study_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
FROM studies s
LEFT JOIN (SELECT DISTINCT study_id FROM locations) l
    ON s.study_id = l.study_id
GROUP BY s.status
ORDER BY without_locations DESC
"""
pd.read_sql(query, conn)

,status,total,without_locations,pct_missing
0,COMPLETED,268,19,7.1
1,UNKNOWN,84,11,13.1
2,WITHDRAWN,19,8,42.1
3,NOT_YET_RECRUITING,16,5,31.3
4,TERMINATED,27,1,3.7
5,WITHHELD,1,1,100.0
6,ACTIVE_NOT_RECRUITING,16,0,0.0
7,AVAILABLE,1,0,0.0
8,ENROLLING_BY_INVITATION,5,0,0.0
9,RECRUITING,62,0,0.0


In [19]:
query = """

Select study_type,count(study_type) from(
SELECT s.nct_id, s.status, s.study_type, s.enrollment, s.start_date, s.title
FROM studies s
LEFT JOIN (SELECT DISTINCT study_id FROM locations) l
    ON s.study_id = l.study_id
WHERE l.study_id IS NULL
  AND s.status = 'COMPLETED'
ORDER BY s.enrollment DESC)
    group by study_type
"""
pd.read_sql(query, conn)

,study_type,count(study_type)
0,INTERVENTIONAL,14
1,OBSERVATIONAL,5


### DEFECT: Missing location data for completed interventional trials

**Observation:** 45 of 500 studies (9%) have no location records.

**Investigation:** Broken down by status and study type:
- 14 (WITHDRAWN/NOT_YET_RECRUITING/WITHHELD) — coherent; trials never activated sites
- 12 (UNKNOWN/TERMINATED) — plausible; stale or truncated records
- 19 COMPLETED — investigated further:
    - 5 OBSERVATIONAL — plausibly explainable (registry/records-based studies
      may have no physical site)
    - **14 INTERVENTIONAL — genuine defect**

**Root cause (hypothesised):** incomplete source-record submission. Sponsors are not
compelled to populate facility data retroactively, so completed trials may retain
sparse site records.

**Impact:** These 14 trials are invisible to geographic analysis (Q5) despite being
the most analytically valuable subset — completed interventional trials carry the
outcome data most relevant to regional performance assessment. Their exclusion biases
geographic findings toward trials with better record-keeping.

**Severity:** Medium (2.8% of cohort; silent exclusion from JOIN-based geographic analysis).

**Recommended resolution:** flag these records in a data-quality dimension; report
geographic coverage explicitly (91%) alongside any regional analysis; escalate to
data stewards for source-record backfill.

In [5]:
query = """
SELECT nct_id, status, study_type, start_date, completion_date, enrollment, title
FROM studies
WHERE start_date IS NOT NULL
  AND (start_date < '1990-01-01' OR start_date > '2031-12-31')
"""
pd.read_sql(query, conn)

,nct_id,status,study_type,start_date,completion_date,enrollment,title
0,NCT00000489,COMPLETED,INTERVENTIONAL,1973-06,1996-12,None,Coronary Artery Surgery Study (CASS)
1,NCT00001191,COMPLETED,INTERVENTIONAL,1983-02-03,2007-12-10,None,The Use of Oral Omeprazole and Intravenous Pan...
